# Outcome Predictor (MDP Architecture)

**What this notebook does:**  
Given a play type (run / pass / punt / field_goal) and game state, predict:

| Head | Task | Applies to |
|------|------|------------|
| `yards_head` | Yards gained — regression | run, pass |
| `turnover_head` | Interception or fumble lost — binary | run, pass |
| `td_head` | Touchdown scored — binary | run, pass |
| `receiver_pos_head` | Targeted position group (WR/TE/RB) — 3-class | pass only |
| `punt_yards_head` | Net punt yards — regression | punt only |
| `punt_blocked_head` | Punt blocked — binary | punt only |
| `fg_result_head` | FG made / missed / blocked — 3-class | field_goal only |

All play types trained together. Each head is masked to only compute loss on relevant rows.

**Before running:** Run on Google Colab, and switch runtime type to T4 GPU on Free or A100 GPU on Pro.

**Output artifacts:**
- `outcome_model.pt`
- `outcome_feature_meta.json`
- `outcome_config.json`
- `outcome_yards_scaler.pkl` — StandardScaler for scrimmage yards
- `outcome_punt_yards_scaler.pkl` — StandardScaler for punt yards

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Switch to T4 GPU runtime before proceeding.')

## 1: Install dependencies

In [ ]:
%%capture
!pip install nflreadpy scikit-learn pandas numpy torch --quiet

## 2: Load play-by-play data

In [ ]:
import nflreadpy as nfl
import pandas as pd
import numpy as np
import pickle, json, os, warnings
warnings.filterwarnings('ignore')

SEASONS = list(range(2015, 2026))
print(f'Loading PBP for seasons: {SEASONS}')
pbp_raw = nfl.load_pbp(SEASONS).to_pandas()
print(f'Loaded: {pbp_raw.shape[0]:,} plays x {pbp_raw.shape[1]} columns')

## 3: Filter to all 4 play types

In [ ]:
KEEP_TYPES = {'run', 'pass', 'punt', 'field_goal'}

df = pbp_raw[
    pbp_raw['play_type'].isin(KEEP_TYPES) &
    pbp_raw['yardline_100'].notna() &
    pbp_raw['score_differential'].notna() &
    pbp_raw['qtr'].notna()
].copy()

df['down'] = df['down'].fillna(4)
df['ydstogo'] = df['ydstogo'].fillna(10)
df['yards_gained'] = df['yards_gained'].fillna(0)
df['shotgun'] = df['shotgun'].fillna(0).astype(int)
df['goal_to_go'] = df['goal_to_go'].fillna(0).astype(int)
df['air_yards'] = df['air_yards'].fillna(0)
df['kick_distance']= df['kick_distance'].fillna(0)
df['return_yards'] = df['return_yards'].fillna(0)

df['punt_net_yards'] = np.where(
    df['play_type'] == 'punt',
    (df['kick_distance'] - df['return_yards']).clip(0, 80),
    0.0
)

print(f'Total plays: {len(df):,}')
print('\nPlay type breakdown:')
for pt in KEEP_TYPES:
    n = (df['play_type'] == pt).sum()
    print(f'  {pt:<12} {n:>8,}')

punts = df[df['play_type'] == 'punt']
print(f'\nPunt sanity check:')
print(f'Avg kick_distance: {punts["kick_distance"].mean():.1f} yds')
print(f'Avg return_yards: {punts["return_yards"].mean():.1f} yds')
print(f'Avg net_yards: {punts["punt_net_yards"].mean():.1f} yds')

## 4: Build target labels

Each head uses a mask so it only trains on relevant rows.

In [ ]:
df['target_yards'] = df['yards_gained'].clip(-10, 50).astype(float)

df['target_turnover'] = (
    (df.get('interception', pd.Series(0, index=df.index)).fillna(0) == 1) |
    (df.get('fumble_lost',  pd.Series(0, index=df.index)).fillna(0) == 1)
).astype(int)

df['target_td'] = df.get('touchdown', pd.Series(0, index=df.index)).fillna(0).astype(int)

pos_col = next((c for c in ['receiver_player_position','receiver_position'] if c in df.columns), None)

def infer_receiver_pos(row):
    if row['play_type'] != 'pass':
        return 0
    if pos_col and pd.notna(row.get(pos_col)):
        p = str(row[pos_col]).upper()
        if p in ('WR',): return 0
        if p in ('TE',): return 1
        if p in ('RB','HB','FB'): return 2
    air = row.get('air_yards', 0) or 0
    if air < 0: return 2
    elif air < 6: return 1
    else: return 0

print('Building receiver position labels (~30s)...')
df['target_receiver_pos'] = df.apply(infer_receiver_pos, axis=1)

df['target_punt_yards'] = df['punt_net_yards'].astype(float)
df['target_punt_blocked'] = df.get('punt_blocked', pd.Series(0, index=df.index)).fillna(0).astype(int)

def fg_result_label(row):
    if row['play_type'] != 'field_goal':
        return 0
    res = str(row.get('field_goal_result', '')).lower()
    if 'made' in res or res == 'good': return 0
    if 'blocked' in res: return 2
    return 1

df['target_fg_result'] = df.apply(fg_result_label, axis=1)

print('\nTarget distributions:')
scrimmage = df[df['play_type'].isin({'run','pass'})]
print(f'Turnovers (run+pass): {scrimmage["target_turnover"].mean()*100:.2f}%')
print(f'TDs (run+pass): {scrimmage["target_td"].mean()*100:.2f}%')
punts = df[df['play_type'] == 'punt']
print(f'Punt blocked rate: {punts["target_punt_blocked"].mean()*100:.2f}%')
print(f'Avg punt net yards: {punts["target_punt_yards"].mean():.1f}')
fgs = df[df['play_type'] == 'field_goal']
print(f'FG made rate: {(fgs["target_fg_result"]==0).mean()*100:.1f}%')
print(f'FG missed rate: {(fgs["target_fg_result"]==1).mean()*100:.1f}%')
print(f'FG blocked rate: {(fgs["target_fg_result"]==2).mean()*100:.1f}%')

## 5: Feature engineering

`feat_play_type` is now 4-class (run/pass/punt/FG). `feat_air_yards` is 0 for non-pass plays.  
`feat_kick_dist` is added for FG. Distance is the single strongest predictor of make probability.

In [ ]:
PLAY_TYPE_MAP = {'run': 0, 'pass': 1, 'punt': 2, 'field_goal': 3}

def dist_bucket(x):
    if x <= 2: return 0
    elif x <= 6: return 1
    elif x <= 10: return 2
    else: return 3

def yard_zone(x):
    if x >= 80: return 0
    elif x >= 60: return 1
    elif x >= 40: return 2
    elif x >= 20: return 3
    elif x >= 5: return 4
    else: return 5

def score_bucket(x):
    if x <= -17: return 0
    elif x <= -7: return 1
    elif x <= 6: return 2
    elif x <= 16: return 3
    else: return 4

def air_yards_bucket(x):
    if x < 0: return 0
    elif x <= 5: return 1
    elif x <= 15: return 2
    else: return 3

def kick_dist_bucket(x):
    if x <= 30: return 0
    elif x <= 40: return 1
    elif x <= 50: return 2
    else: return 3

df['feat_play_type'] = df['play_type'].map(PLAY_TYPE_MAP)
df['feat_down'] = df['down'].astype(int).clip(1, 4) - 1
df['feat_dist'] = df['ydstogo'].apply(dist_bucket)
df['feat_zone'] = df['yardline_100'].apply(yard_zone)
df['feat_score'] = df['score_differential'].apply(score_bucket)
df['feat_qtr'] = (df['qtr'].clip(1, 5) - 1).astype(int)
df['feat_shotgun'] = df['shotgun']
df['feat_goal_to_go'] = df['goal_to_go']
df['feat_air_yards'] = df['air_yards'].apply(air_yards_bucket)
df['feat_kick_dist'] = df['kick_distance'].apply(kick_dist_bucket)

FEATURE_COLS = [
    'feat_play_type', 'feat_down', 'feat_dist', 'feat_zone',
    'feat_score', 'feat_qtr', 'feat_shotgun', 'feat_goal_to_go',
    'feat_air_yards', 'feat_kick_dist',
]

FEAT_CARDINALITY = {
    'feat_play_type': 4,
    'feat_down': 4,
    'feat_dist': 4,
    'feat_zone': 6,
    'feat_score': 5,
    'feat_qtr': 5,
    'feat_shotgun': 2,
    'feat_goal_to_go': 2,
    'feat_air_yards': 4,
    'feat_kick_dist': 4,
}

print('Feature engineering done.')
print(df[FEATURE_COLS].head(3))

## 6: Scale regression targets + train/val split

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = df[FEATURE_COLS].values.astype(np.int64)

y_yards = df['target_yards'].values.astype(np.float32)
y_punt_yards = df['target_punt_yards'].values.astype(np.float32)

y_turnover = df['target_turnover'].values.astype(np.int64)
y_td = df['target_td'].values.astype(np.int64)
y_rec_pos = df['target_receiver_pos'].values.astype(np.int64)
y_punt_blocked = df['target_punt_blocked'].values.astype(np.int64)
y_fg_result = df['target_fg_result'].values.astype(np.int64)

yards_scaler = StandardScaler()
y_yards_sc = yards_scaler.fit_transform(y_yards.reshape(-1,1)).flatten().astype(np.float32)

punt_mask_all = df['play_type'].values == 'punt'
punt_yards_scaler = StandardScaler()
punt_yards_scaler.fit(y_punt_yards[punt_mask_all].reshape(-1,1))
y_punt_yards_sc = punt_yards_scaler.transform(y_punt_yards.reshape(-1,1)).flatten().astype(np.float32)

print(f'Yards scaler mean: {yards_scaler.mean_[0]:.2f}  std: {yards_scaler.scale_[0]:.2f}')
print(f'Punt yards scaler mean: {punt_yards_scaler.mean_[0]:.2f}  std: {punt_yards_scaler.scale_[0]:.2f}')

idx = np.arange(len(X))
idx_train, idx_val = train_test_split(idx, test_size=0.10, random_state=42)

def split(arr): 
    return arr[idx_train], arr[idx_val]

X_tr, X_vl = split(X)
yw_tr, yw_vl = split(y_yards_sc)
yp_tr, yp_vl = split(y_punt_yards_sc)
yt_tr, yt_vl = split(y_turnover)
ytd_tr, ytd_vl = split(y_td)
yrp_tr, yrp_vl = split(y_rec_pos)
ypb_tr, ypb_vl = split(y_punt_blocked)
yfg_tr, yfg_vl = split(y_fg_result)

print(f'Train: {len(X_tr):,}  Val: {len(X_vl):,}')

## 7: Define the multi-task model

Shared encoder → 7 heads. Each head masked to its relevant play type during loss computation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

EMB_DIM = 8
HIDDEN = 512
DROPOUT = 0.3


class OutcomeMLP(nn.Module):
    def __init__(self, feat_cardinality: dict, emb_dim: int, hidden: int):
        super().__init__()
        self.feat_names = list(feat_cardinality.keys())
        self.embeddings = nn.ModuleList([
            nn.Embedding(card, emb_dim) for card in feat_cardinality.values()
        ])
        in_dim = len(feat_cardinality) * emb_dim

        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden), 
            nn.LayerNorm(hidden), 
            nn.SiLU(), 
            nn.Dropout(DROPOUT),
            nn.Linear(hidden, hidden), 
            nn.LayerNorm(hidden), nn.SiLU(), 
            nn.Dropout(DROPOUT),
            nn.Linear(hidden, hidden // 2), 
            nn.LayerNorm(hidden // 2), 
            nn.SiLU(),
        )
        h = hidden // 2
        self.yards_head = nn.Linear(h, 1)
        self.turnover_head = nn.Linear(h, 2)
        self.td_head = nn.Linear(h, 2)
        self.receiver_pos_head = nn.Linear(h, 3)
        self.punt_yards_head = nn.Linear(h, 1)
        self.punt_blocked_head = nn.Linear(h, 2)
        self.fg_result_head = nn.Linear(h, 3)

    def forward(self, x: torch.Tensor):
        embs = [self.embeddings[i](x[:, i]) for i in range(len(self.feat_names))]
        h = self.encoder(torch.cat(embs, dim=-1))
        return (
            self.yards_head(h).squeeze(-1),
            self.turnover_head(h),
            self.td_head(h),
            self.receiver_pos_head(h),
            self.punt_yards_head(h).squeeze(-1),
            self.punt_blocked_head(h),
            self.fg_result_head(h),
        )

    def predict(self, x: torch.Tensor, yards_scaler, punt_yards_scaler):
        self.eval()
        with torch.no_grad():
            y_hat, to_l, td_l, rp_l, py_hat, pb_l, fg_l = self.forward(x)
        yards = yards_scaler.inverse_transform(y_hat.cpu().numpy().reshape(-1,1)).flatten()
        punt_yards = punt_yards_scaler.inverse_transform(py_hat.cpu().numpy().reshape(-1,1)).flatten()
        to_prob = torch.softmax(to_l, -1)[:, 1].cpu().numpy()
        td_prob = torch.softmax(td_l, -1)[:, 1].cpu().numpy()
        rp_probs = torch.softmax(rp_l, -1).cpu().numpy()
        pb_prob = torch.softmax(pb_l, -1)[:, 1].cpu().numpy()
        fg_probs = torch.softmax(fg_l, -1).cpu().numpy()
        return yards, to_prob, td_prob, rp_probs, punt_yards, pb_prob, fg_probs


model = OutcomeMLP(FEAT_CARDINALITY, EMB_DIM, HIDDEN).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 8: Train

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

EPOCHS = 60
BATCH_SIZE = 4096
LR = 3e-3

W_YARDS = 1.0
W_TURNOVER = 2.0
W_TD = 2.0
W_REC_POS = 0.5
W_PUNT_YARDS = 1.0
W_PUNT_BLOCK = 3.0
W_FG_RESULT = 2.0

def to_gpu(arr, dtype=torch.LongTensor):
    return dtype(arr).to(DEVICE)

X_tr_t = to_gpu(X_tr)
X_vl_t = to_gpu(X_vl)
yw_tr_t = to_gpu(yw_tr, torch.FloatTensor)
yw_vl_t = to_gpu(yw_vl, torch.FloatTensor)
yp_tr_t = to_gpu(yp_tr, torch.FloatTensor)
yp_vl_t = to_gpu(yp_vl, torch.FloatTensor)
yt_tr_t = to_gpu(yt_tr)
yt_vl_t = to_gpu(yt_vl)
ytd_tr_t = to_gpu(ytd_tr)
ytd_vl_t = to_gpu(ytd_vl)
yrp_tr_t = to_gpu(yrp_tr)
yrp_vl_t = to_gpu(yrp_vl)
ypb_tr_t = to_gpu(ypb_tr)
ypb_vl_t = to_gpu(ypb_vl)
yfg_tr_t = to_gpu(yfg_tr)
yfg_vl_t = to_gpu(yfg_vl)

train_ds = TensorDataset(X_tr_t, yw_tr_t, yp_tr_t, yt_tr_t, ytd_tr_t, yrp_tr_t, ypb_tr_t, yfg_tr_t)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, steps_per_epoch=len(train_dl), epochs=EPOCHS
)

best_val_loss = float('inf')

def masked_loss(loss_fn, logits, targets, mask):
    if mask.sum() == 0:
        return torch.tensor(0.0, device=DEVICE)
    return loss_fn(logits[mask], targets[mask])

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, ywb, ypb_y, ytb, ytdb, yrpb, ypbb, yfgb in train_dl:
        optimizer.zero_grad()
        y_hat, to_l, td_l, rp_l, py_hat, pb_l, fg_l = model(xb)

        pt = xb[:, 0]
        scrimmage_m = (pt == 0) | (pt == 1)
        pass_m = (pt == 1)
        punt_m = (pt == 2)
        fg_m = (pt == 3)

        loss = (
            W_YARDS * masked_loss(F.mse_loss, y_hat, ywb, scrimmage_m) +
            W_TURNOVER * masked_loss(F.cross_entropy, to_l, ytb, scrimmage_m) +
            W_TD * masked_loss(F.cross_entropy, td_l, ytdb, scrimmage_m) +
            W_REC_POS * masked_loss(F.cross_entropy, rp_l, yrpb, pass_m)      +
            W_PUNT_YARDS * masked_loss(F.mse_loss, py_hat, ypb_y, punt_m)     +
            W_PUNT_BLOCK * masked_loss(F.cross_entropy, pb_l, ypbb, punt_m)      +
            W_FG_RESULT * masked_loss(F.cross_entropy, fg_l, yfgb, fg_m)
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    if epoch % 10 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            y_hat_v, to_v, td_v, rp_v, py_v, pb_v, fg_v = model(X_vl_t)
            pt_v = X_vl_t[:, 0]
            sc_v = (pt_v==0)|(pt_v==1); pa_v = pt_v==1; pu_v = pt_v==2; fg_mask_v = pt_v==3

            vl = (
                W_YARDS * masked_loss(F.mse_loss, y_hat_v, yw_vl_t, sc_v).item() +
                W_TURNOVER * masked_loss(F.cross_entropy, to_v, yt_vl_t, sc_v).item() +
                W_TD * masked_loss(F.cross_entropy, td_v, ytd_vl_t,sc_v).item() +
                W_REC_POS * masked_loss(F.cross_entropy, rp_v, yrp_vl_t,pa_v).item() +
                W_PUNT_YARDS * masked_loss(F.mse_loss, py_v, yp_vl_t, pu_v).item() +
                W_PUNT_BLOCK * masked_loss(F.cross_entropy, pb_v, ypb_vl_t,pu_v).item() +
                W_FG_RESULT * masked_loss(F.cross_entropy, fg_v, yfg_vl_t,fg_mask_v).item()
            )
            fg_acc = (fg_v[fg_mask_v].argmax(-1)==yfg_vl_t[fg_mask_v]).float().mean().item() if fg_mask_v.sum()>0 else 0
            to_acc = (to_v[sc_v].argmax(-1)==yt_vl_t[sc_v]).float().mean().item()

        print(f'Epoch {epoch:3d} | train {total_loss/len(train_dl):.4f} '
              f'| val {vl:.4f} | to_acc {to_acc*100:.1f}% | fg_acc {fg_acc*100:.1f}%')

        if vl < best_val_loss:
            best_val_loss = vl
            torch.save(model.state_dict(), '/tmp/outcome_model_best.pt')

print(f'\nBest val loss: {best_val_loss:.4f}')

## 9: Calibration check

In [ ]:
model.load_state_dict(torch.load('/tmp/outcome_model_best.pt', map_location=DEVICE))
model.eval()

with torch.no_grad():
    y_hat_v, to_v, td_v, rp_v, py_v, pb_v, fg_v = model(X_vl_t)

pt_v = X_vl_t[:, 0].cpu().numpy()
sc_m = (pt_v==0)|(pt_v==1) 
pu_m = pt_v==2
fg_m_np = pt_v==3

print('=== Scrimmage yards ===')
pred_y = yards_scaler.inverse_transform(y_hat_v.cpu().numpy().reshape(-1,1)).flatten()
true_y = y_yards[idx_val]
for label, mask in [('All', sc_m), ('Run', pt_v==0), ('Pass', pt_v==1)]:
    print(f'{label:<5} true mean={true_y[mask].mean():.2f}  pred mean={pred_y[mask].mean():.2f}')

print('\n=== Punt yards ===')
pred_py = punt_yards_scaler.inverse_transform(py_v.cpu().numpy().reshape(-1,1)).flatten()
true_py = y_punt_yards[idx_val]
print(f'True mean={true_py[pu_m].mean():.1f}  Pred mean={pred_py[pu_m].mean():.1f}')

print('\n=== FG result ===')
fg_preds = fg_v[torch.BoolTensor(fg_m_np).to(DEVICE)]
fg_probs = torch.softmax(fg_preds, -1).cpu().numpy()
true_fg = y_fg_result[idx_val][fg_m_np]
for i, label in [(0,'Made'),(1,'Missed'),(2,'Blocked')]:
    print(f'{label:<8} pred={fg_probs[:,i].mean()*100:.1f}%  true={(true_fg==i).mean()*100:.1f}%')

print('\n=== Scenario inference ===')
SCENARIOS = [
    ('Run 1st&10 own25 tied Q1', [0, 0, 2, 1, 2, 0, 0, 0, 0, 0]),
    ('Pass 3rd&8 own30 trail Q4', [1, 2, 2, 1, 1, 3, 1, 0, 2, 0]),
    ('Punt 4th&10 own30 tied Q2', [2, 3, 2, 1, 2, 1, 0, 0, 0, 2]),
    ('FG 4th&5 opp25 tied Q4', [3, 3, 1, 4, 2, 3, 0, 0, 0, 1]),
    ('FG 4th&3 opp45 tied Q4 (long)', [3, 3, 0, 3, 2, 3, 0, 0, 0, 3]),
    ('Run 2nd&1 opp1 GTG lead Q3', [0, 1, 0, 5, 3, 2, 0, 1, 0, 0]),
]
print(f'{"Scenario":<38} yards  TO%   TD%   | punt_yds | FG:made/miss/blk')
print('-'*90)
for label, feats in SCENARIOS:
    x = torch.LongTensor([feats]).to(DEVICE)
    yards, to_p, td_p, rp_p, punt_y, pb_p, fg_p = model.predict(x, yards_scaler, punt_yards_scaler)
    pt = feats[0]
    print(f'{label:<38} {yards[0]:5.1f}  {to_p[0]*100:4.1f}%  {td_p[0]*100:4.1f}%  | '
          f'{punt_y[0]:6.1f}yd | {fg_p[0][0]*100:.0f}%/{fg_p[0][1]*100:.0f}%/{fg_p[0][2]*100:.0f}%')

## 10: Save all artifacts

In [ ]:
import shutil

SAVE_DIR = '/tmp/outcome_model_weights'
os.makedirs(SAVE_DIR, exist_ok=True)

shutil.copy('/tmp/outcome_model_best.pt', f'{SAVE_DIR}/outcome_model.pt')
print('Saved outcome_model.pt')

with open(f'{SAVE_DIR}/outcome_yards_scaler.pkl', 'wb') as f:
    pickle.dump(yards_scaler, f)
with open(f'{SAVE_DIR}/outcome_punt_yards_scaler.pkl', 'wb') as f:
    pickle.dump(punt_yards_scaler, f)
print('Saved scalers')

feature_meta = {
    'feature_cols': FEATURE_COLS,
    'feat_cardinality': FEAT_CARDINALITY,
    'play_type_map': PLAY_TYPE_MAP,
    'receiver_pos_classes': {0:'WR', 1:'TE', 2:'RB'},
    'fg_result_classes': {0:'made', 1:'missed', 2:'blocked'},
    'bucket_definitions': {
        'feat_play_type': {'values': {0:'run',1:'pass',2:'punt',3:'field_goal'}},
        'feat_down': {'values': {0:'1st',1:'2nd',2:'3rd',3:'4th'}},
        'feat_dist': {'values': {0:'short(1-2)',1:'medium(3-6)',2:'long(7-10)',3:'very_long(11+)'}},
        'feat_zone': {'values': {0:'own_deep',1:'own_mid',2:'midfield',3:'opp_mid',4:'red_zone',5:'goal_line'}},
        'feat_score': {'values': {0:'large_deficit',1:'deficit',2:'close',3:'lead',4:'large_lead'}},
        'feat_qtr': {'values': {0:'Q1',1:'Q2',2:'Q3',3:'Q4',4:'OT'}},
        'feat_shotgun': {'values': {0:'under_center',1:'shotgun'}},
        'feat_goal_to_go': {'values': {0:'normal',1:'goal_to_go'}},
        'feat_air_yards': {'values': {0:'screen/behind',1:'short(0-5)',2:'medium(6-15)',3:'deep(16+)'}},
        'feat_kick_dist': {'values': {0:'chip(<=30)',1:'short(31-40)',2:'medium(41-50)',3:'long(51+)'}},
    }
}
with open(f'{SAVE_DIR}/outcome_feature_meta.json', 'w') as f:
    json.dump(feature_meta, f, indent=2)
print('Saved outcome_feature_meta.json')

model_config = {
    'emb_dim': EMB_DIM,
    'hidden': HIDDEN,
    'dropout': DROPOUT,
    'n_features': len(FEATURE_COLS),
    'best_val_loss': best_val_loss,
    'training_seasons': SEASONS,
    'yards_clip': [-10, 50],
    'punt_yards_clip': [0, 80],
}
with open(f'{SAVE_DIR}/outcome_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)
print('Saved outcome_config.json')

print(f'\nAll artifacts in {SAVE_DIR}:')
for fname in sorted(os.listdir(SAVE_DIR)):
    print(f'{fname:<48} {os.path.getsize(f"{SAVE_DIR}/{fname}")/1024:.1f} KB')

## 11: Download

In [ ]:
from google.colab import files

zip_path = '/tmp/outcome_model_weights_export'
shutil.make_archive(zip_path, 'zip', SAVE_DIR)
files.download(f'{zip_path}.zip')
print('Download started. Extract into backend/python_backend/outcome_model_weights/')

## Appendix: Quick inference test

In [ ]:
with open(f'{SAVE_DIR}/outcome_yards_scaler.pkl', 'rb') as f:
    ys_loaded = pickle.load(f)
with open(f'{SAVE_DIR}/outcome_punt_yards_scaler.pkl', 'rb') as f:
    pys_loaded = pickle.load(f)

model_loaded = OutcomeMLP(FEAT_CARDINALITY, EMB_DIM, HIDDEN).to(DEVICE)
model_loaded.load_state_dict(torch.load(f'{SAVE_DIR}/outcome_model.pt', map_location=DEVICE))
print('Cold-load successful.')

x = torch.LongTensor([[3, 3, 2, 4, 2, 3, 0, 0, 0, 1]]).to(DEVICE)
yards, to_p, td_p, rp_p, punt_y, pb_p, fg_p = model_loaded.predict(x, ys_loaded, pys_loaded)
print(f'\n40-yd FG attempt — made={fg_p[0][0]*100:.1f}%  missed={fg_p[0][1]*100:.1f}%  blocked={fg_p[0][2]*100:.1f}%')

x = torch.LongTensor([[2, 3, 3, 1, 2, 1, 0, 0, 0, 2]]).to(DEVICE)
yards, to_p, td_p, rp_p, punt_y, pb_p, fg_p = model_loaded.predict(x, ys_loaded, pys_loaded)
print(f'Punt own 30      — net yards={punt_y[0]:.1f}  blocked={pb_p[0]*100:.2f}%')